# Code to deal with Prospector output files! Requires own environment

In [2]:
import os
import glob
import numpy as np
import pickle as pkl
import pandas as pd
import prospect.io.read_results as reader
from prospect.utils.plotting import get_percentiles, get_best

from astropy.io import fits
from astropy.cosmology import WMAP9 as cosmo
from prospect.models.transforms import logsfr_ratios_to_sfrs
from prospect.sources import FastStepBasis
from astropy.cosmology import Planck18 as cosmo
from prospect.models.sedmodel import PolySpecModel, SpecModel



Load Prospector outputs

In [3]:
def load_prospector_results(galaxy_id, prosp_dir):
    """Function to load Prospector result .h5 files and disect its 
    data structure to be used for easy plotting.

    Args:
        galaxy_id (int): The ID of the galaxy for which to load results
        prosp_dir (str): The directory containing the Prospector output files

    Returns:
        dict: A comprehensive dictionary storing the parameters samples, MAP values and quantiles
    """
    
    # Load the h5 file for the given galaxy ID
    h5_files = glob.glob(os.path.join(prosp_dir, f'*{galaxy_id}*.h5'))
    
    try:
        h5_file = h5_files[0]
        print(f"Loading Prospector output file: {h5_file}")
    except IndexError:
        print(f"No PROSPECTOR results found for objid {galaxy_id}.")
        return None

    # Load PROSPECTOR results
    results, obs, model = reader.results_from(h5_file)
        
    # Now we have to exclude the last 3 parameters from the fit
    map_parameters = get_best(results)
    
    # Extract labels for parameters that were "free" (fitted)
    labels = map_parameters[0]

    # Build the MAP dictionary
    MAP = {}
    for a,b in zip(map_parameters[0], map_parameters[1]):
        MAP[a] = b
    
    # Extract chains, weights and the MAP index
    chain = results['chain']
    weights = results['weights']
    imax = np.argmax(results['lnprobability'])
    
    data = {
            'meta': {'labels': map_parameters[0], 'map_idx': imax, 'weights': weights},
            'params': {}
        }

    perc = get_percentiles(results, [16, 50, 84])    
    
    for i, name in enumerate(map_parameters[0]):
            # Use the flattened chain for statistics
            param_samples = chain[:, i]
            
            data['params'][name] = {
                'samples': param_samples,
                'map': MAP[name],   # Use the value from the best vector directly
                'q16': perc[name][0],
                'q50': perc[name][1],
                'q84': perc[name][2]
            }
    
    return data

phot_table = './Phot_Table_MIRI.fits'
with fits.open(phot_table) as hdul:
    galaxy_ids = hdul[1].data['ID']

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'

data = load_prospector_results(12717, prosp_dir)
print(data)

Loading Prospector output file: /Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/output_12717_onlyphot_wMIRI_mcmc.h5
{'meta': {'labels': ['zred', 'logzsol', 'f_outlier_phot', 'logmass', 'logsfr_ratios_1', 'logsfr_ratios_2', 'logsfr_ratios_3', 'logsfr_ratios_4', 'logsfr_ratios_5', 'logsfr_ratios_6', 'logsfr_ratios_7', 'logsfr_ratios_8', 'logsfr_ratios_9', 'logsfr_ratios_10', 'logsfr_ratios_11', 'logsfr_ratios_12', 'logsfr_ratios_13', 'dust2', 'dust_index', 'dust1_fraction', 'duste_umin', 'duste_qpah', 'duste_gamma', 'gas_logz', 'gas_logu'], 'map_idx': 82862, 'weights': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
       3.17454934e-06, 3.22844983e-06, 3.30022357e-06])}, 'params': {'zred': {'samples': array([1.77120433, 1.76234369, 1.76761057, ..., 1.76386075, 1.7638457 ,
       1.76384719]), 'map': 1.7643080889081182, 'q16': 1.7637549624468665, 'q50': 1.764066787171568, 'q84': 1.764441854754242}, 'logzsol': {'samples': array([-0.73473232, -0.197018